In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import json

import model_inference


In [2]:
# Build a .py script that takes a snapshot date, loads a model artefact and make an inference and save to datamart

## set up pyspark session

In [3]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/19 20:00:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [4]:
snapshot_date_str = "2024-01-01"
model_name = "xgboostv1.json"


In [5]:
config = {}
config["snapshot_date_str"] = snapshot_date_str
config["snapshot_date"] = datetime.strptime(config["snapshot_date_str"], "%Y-%m-%d")
config["model_name"] = model_name
config["model_bank_directory"] = "model_bank/"
config["model_artefact_filepath"] = config["model_bank_directory"] + config["model_name"]

pprint.pprint(config)

{'model_artefact_filepath': 'model_bank/xgboostv1.json',
 'model_bank_directory': 'model_bank/',
 'model_name': 'xgboostv1.json',
 'snapshot_date': datetime.datetime(2024, 1, 1, 0, 0),
 'snapshot_date_str': '2024-01-01'}


## load model artefact from model bank

In [6]:
# Load the model from the pickle file
with open(config["model_artefact_filepath"], 'rb') as file:
    model_artefact = json.load(file)

print("Model loaded successfully! " + config["model_artefact_filepath"])

Model loaded successfully! model_bank/xgboostv1.json


## load feature store

In [7]:
import glob, os
from pyspark.sql.functions import col

feature_location = "datamart/gold/feature/"

files_list = glob.glob(os.path.join(feature_location, "*.parquet"))
if not files_list:
    raise FileNotFoundError(f"No .parquet files in {feature_location}")
print("Reading these files:", files_list)

features_store_sdf = spark.read.parquet(*files_list)
print("TOTAL ROWS IN FEATURE STORE:", features_store_sdf.count())

features_store_sdf.printSchema()
features_store_sdf.show(5, truncate=False)

features_sdf = features_store_sdf.filter(
    col("snapshot_date") == config["snapshot_date"]
)
print(f"Rows for {config['snapshot_date'].date()}:", features_sdf.count())

features_pdf = features_sdf.toPandas()
features_pdf

Reading these files: ['datamart/gold/feature/gold_feature_store_2024_01_01.parquet', 'datamart/gold/feature/gold_feature_store_2023_04_01.parquet', 'datamart/gold/feature/gold_feature_store_2023_09_01.parquet', 'datamart/gold/feature/gold_feature_store_2024_10_01.parquet', 'datamart/gold/feature/gold_feature_store_2024_09_01.parquet', 'datamart/gold/feature/gold_feature_store_2023_10_01.parquet', 'datamart/gold/feature/gold_feature_store_2024_04_01.parquet', 'datamart/gold/feature/gold_feature_store_2023_01_01.parquet', 'datamart/gold/feature/gold_feature_store_2023_06_01.parquet', 'datamart/gold/feature/gold_feature_store_2024_03_01.parquet', 'datamart/gold/feature/gold_feature_store_2024_12_01.parquet', 'datamart/gold/feature/gold_feature_store_2023_12_01.parquet', 'datamart/gold/feature/gold_feature_store_2023_03_01.parquet', 'datamart/gold/feature/gold_feature_store_2024_06_01.parquet', 'datamart/gold/feature/gold_feature_store_2023_11_01.parquet', 'datamart/gold/feature/gold_featu

,Customer_ID,fe_1,fe_2,fe_3,fe_4,fe_5,fe_6,fe_7,fe_8,fe_9,...,Auto_Loan_count,Credit_Builder_Loan_count,Debt_Consolidation_Loan_count,Home_Equity_Loan_count,Mortgage_Loan_count,Not_Specified_count,Payday_Loan_count,Personal_Loan_count,Student_Loan_count,Unknown_count
0,CUS_0x1037,239.0,140.0,-24.0,265.0,2.0,-32.0,147.0,-38.0,280.0,...,2,1,0,0,1,0,0,0,0,0
1,CUS_0x1069,-15.0,137.0,35.0,-9.0,124.0,23.0,-33.0,-66.0,134.0,...,1,0,0,0,0,1,0,1,0,0
2,CUS_0x114a,361.0,58.0,-37.0,150.0,17.0,40.0,189.0,-64.0,119.0,...,0,0,0,1,0,0,0,0,1,0
3,CUS_0x1184,96.0,231.0,-41.0,20.0,147.0,217.0,158.0,-45.0,8.0,...,0,0,0,0,1,0,1,0,1,0
4,CUS_0x1297,54.0,294.0,-89.0,-43.0,288.0,355.0,57.0,-33.0,157.0,...,0,1,0,1,1,0,3,2,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8969,CUS_0xdf6,141.0,154.0,89.0,149.0,155.0,114.0,-94.0,277.0,93.0,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,0
8970,CUS_0xe23,96.0,52.0,242.0,123.0,207.0,155.0,147.0,189.0,121.0,...,0,0,0,1,0,0,0,0,0,0
8971,CUS_0xe4e,75.0,-7.0,-42.0,60.0,57.0,237.0,12.0,90.0,225.0,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,0
8972,CUS_0xedd,138.0,153.0,38.0,306.0,282.0,171.0,-23.0,102.0,281.0,...,0,1,0,0,0,0,1,1,1,0


## preprocess data for modeling

In [8]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [9]:
X_raw = features_pdf.drop(columns=["Name", "SSN"])

cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_raw.select_dtypes(include=["number"]).columns.tolist()

print("Categorical columns to encode:", cat_cols)
print("Numeric columns to scale:  ", num_cols)

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_cat = pd.DataFrame(
    ohe.fit_transform(X_raw[cat_cols]),
    columns=ohe.get_feature_names_out(cat_cols),
    index=X_raw.index
)

scaler = StandardScaler()
X_num_scaled = pd.DataFrame(
    scaler.fit_transform(X_raw[num_cols]),
    columns=num_cols,
    index=X_raw.index
)

X_processed = pd.concat([X_num_scaled, X_cat], axis=1)
print("Final preprocessed shape:", X_processed.shape)

print(X_processed.head())


Categorical columns to encode: ['Customer_ID', 'snapshot_date', 'Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']
Numeric columns to scale:   ['fe_1', 'fe_2', 'fe_3', 'fe_4', 'fe_5', 'fe_6', 'fe_7', 'fe_8', 'fe_9', 'fe_10', 'fe_11', 'fe_12', 'fe_13', 'fe_14', 'fe_15', 'fe_16', 'fe_17', 'fe_18', 'fe_19', 'fe_20', 'fe_1_5_mean', 'Age_num', 'Age_missing', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance', 'days_overdue_per_late_payment', 'Credit_History_Age_num', 'debt_to_income_ratio', 'monthly_repayment_to_income', 'credit_inquiries_per_year', 'Auto_Loan_count', 'Credit_Builder_Loan_count', 'Debt_Consolidation_Loan_count', 'Home_Equity_Loan_count', 'Mortgage_Loan_count', 'Not_Specified_count

## model prediction inference

In [10]:
import xgboost as xgb
import numpy as np
import pandas as pd

label_map = {
    "Annual_Income":                 "Annual Income ($)",
    "Num_Bank_Accounts":             "No. of Bank Accounts",
    "Num_Credit_Card":               "No. of Credit Cards Owned",
    "Interest_Rate":                 "Interest Rate (%)",
    "Num_of_Loan":                   "Total No. of Loans",
    "Delay_from_due_date":           "Payment Delay (days)",
    "Num_of_Delayed_Payment":        "No. of Delayed Payments",
    "Num_Credit_Inquiries":          "Total Credit Inquiries",
    "Outstanding_Debt":              "Outstanding Debt",
    "days_overdue_per_late_payment": "Avg Days Overdue per Late Payment",
    "Credit_History_Age_num":        "Credit History Age (yrs)",
    "debt_to_income_ratio":          "Debt-to-Income Ratio",
    "monthly_repayment_to_income":   "Repayment-to-Income Ratio",
    "credit_inquiries_per_year":     "Credit Inquiries / Year",
    "Mortgage_Loan_count":           "No. of Mortgage Loans",
    "Student_Loan_count":            "No. of Student Loans",
    "Credit_Mix_Bad":                "Poor Credit Profile",
    "Payment_of_Min_Amount_No":      "No Minimum Payment Made",
    "Payment_of_Min_Amount_Yes":     "Minimum Payment Made",
}

selected_features = list(label_map.keys())
X_sel = X_processed[selected_features]
print("Shape after selecting 19 features:", X_sel.shape)  # should be (n_rows, 19)

model_filepath = os.path.join("model_bank", "xgboostv1.json")
bst = xgb.Booster()
bst.load_model(model_filepath)
print("Loaded XGBoost JSON model from", model_filepath)

dmat = xgb.DMatrix(X_sel.values)
y_pred_proba = bst.predict(dmat)
results_df = features_pdf[["Customer_ID", "snapshot_date"]].copy()
results_df["model_name"]        = "xgboostv1.json"
results_df["model_predictions"] = y_pred_proba

print(results_df.head())

Shape after selecting 19 features: (8974, 19)
Loaded XGBoost JSON model from model_bank/xgboostv1.json
  Customer_ID snapshot_date      model_name  model_predictions
0  CUS_0x1037    2024-01-01  xgboostv1.json           0.251601
1  CUS_0x1069    2024-01-01  xgboostv1.json           0.266159
2  CUS_0x114a    2024-01-01  xgboostv1.json           0.338661
3  CUS_0x1184    2024-01-01  xgboostv1.json           0.310813
4  CUS_0x1297    2024-01-01  xgboostv1.json           0.513788


## save model inference to datamart gold table

In [11]:
gold_directory = f"datamart/gold/model_predictions/{config["model_name"][:-4]}/"
print(gold_directory)

if not os.path.exists(gold_directory):
    os.makedirs(gold_directory)

# save gold table - IRL connect to database to write
partition_name = config["model_name"][:-4] + "_predictions_" + snapshot_date_str.replace('-','_') + '.parquet'
filepath = gold_directory + partition_name
spark.createDataFrame(results_df).write.mode("overwrite").parquet(filepath)
# df.toPandas().to_parquet(filepath,
#           compression='gzip')
print('saved to:', filepath)

datamart/gold/model_predictions/xgboostv1./
saved to: datamart/gold/model_predictions/xgboostv1./xgboostv1._predictions_2024_01_01.parquet


## backfill

In [12]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2024-12-01"

In [13]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)


In [14]:
for snapshot_date in dates_str_lst:
    print(snapshot_date)
    model_inference.main(snapshot_date, model_name)

2023-01-01


---starting job---


{'model_artefact_filepath': 'model_bank/xgboostv1.json',
 'model_bank_directory': 'model_bank/',
 'model_name': 'xgboostv1.json',
 'snapshot_date': datetime.datetime(2023, 1, 1, 0, 0),
 'snapshot_date_str': '2023-01-01'}
Model loaded successfully! model_bank/xgboostv1.json
Reading these files: ['datamart/gold/feature/gold_feature_store_2024_01_01.parquet', 'datamart/gold/feature/gold_feature_store_2023_04_01.parquet', 'datamart/gold/feature/gold_feature_store_2023_09_01.parquet', 'datamart/gold/feature/gold_feature_store_2024_10_01.parquet', 'datamart/gold/feature/gold_feature_store_2024_09_01.parquet', 'datamart/gold/feature/gold_feature_store_2023_10_01.parquet', 'datamart/gold/feature/gold_feature_store_2024_04_01.parquet', 'datamart/gold/feature/gold_feature_store_2023_01_01.parquet', 'datamart/gold/feature/gold_feature_store_2023_06_01.parquet', 'datamart/gold/feature/gold_feature_store_2024_03_01.parquet', 'datamart/gold/feature/gold_feature_stor

## Check datamart

In [15]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

In [16]:
folder_path = "datamart/gold/model_predictions/xgboostv1./"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",df.count())

df.show()

row_count: 218376
+-----------+-------------+--------------+-------------------+
|Customer_ID|snapshot_date|    model_name|  model_predictions|
+-----------+-------------+--------------+-------------------+
| CUS_0x1182|   2024-07-01|xgboostv1.json| 0.1860024631023407|
| CUS_0x11c6|   2024-07-01|xgboostv1.json| 0.3053717017173767|
| CUS_0x1246|   2024-07-01|xgboostv1.json| 0.5012264847755432|
| CUS_0x12bf|   2024-07-01|xgboostv1.json|0.28723040223121643|
| CUS_0x12ec|   2024-07-01|xgboostv1.json| 0.7045090198516846|
| CUS_0x1330|   2024-07-01|xgboostv1.json|0.29947397112846375|
| CUS_0x141c|   2024-07-01|xgboostv1.json|0.26459264755249023|
| CUS_0x1430|   2024-07-01|xgboostv1.json|0.20456722378730774|
| CUS_0x14d1|   2024-07-01|xgboostv1.json| 0.5488044619560242|
| CUS_0x14de|   2024-07-01|xgboostv1.json|  0.512083113193512|
| CUS_0x14e5|   2024-07-01|xgboostv1.json| 0.4243125319480896|
| CUS_0x1619|   2024-07-01|xgboostv1.json|0.23212994635105133|
| CUS_0x1666|   2024-07-01|xgboostv1.